In [5]:
import pandas as pd
from datetime import datetime

with open('output.nmea', 'r') as file:
    nmea_data = file.readlines()

# Sort '' if it's existed in the data
clean_data = [line.split('] ')[-1].strip() if ']' in line else line.strip() for line in nmea_data]

In [ ]:
# @title Parsing information
def parse_nmea(lines):
    gps_results = []
    #
    current_elevation = "0.0"
    current_sats = "0"

    for line in lines:
        parts = line.split(',')

        # Take the heights and satellites from GGA
        if parts[0] == '$GPGGA':
            current_sats = parts[7]
            current_elevation = parts[9]

        # Take the information from RMC
        if parts[0] == '$GPRMC':
            time_utc = parts[1]
            speed_knots = float(parts[7]) if parts[7] else 0.0
            date_str = parts[9]

            gps_results.append({
                "Date": f"20{date_str[4:6]}-{date_str[2:4]}-{date_str[:2]}",
                "Time (UTC)": f"{time_utc[:2]}:{time_utc[2:4]}:{time_utc[4:6]}",
                "Speed (km/h)": round(speed_knots * 1.852, 2),
                "Elevation (M)": current_elevation,
                "Satellites": current_sats,        
                "Status": "Valid" if parts[2] == 'A' else "Invalid"
            })
    return gps_results

parsed_info = parse_nmea(clean_data)

In [17]:
df = pd.DataFrame(parsed_info)

if not df.empty:
    latest_time = df.iloc[-1]['Time (UTC)']
    latest_date = df.iloc[-1]['Date']
    print(f"Data Display")

df.head(10)

Data Display


,Date,Time (UTC),Speed (km/h),Elevation (M),Satellites,Status
0,2026-04-19,13:35:11,1491.60,0.0,12,Valid
1,2026-04-19,13:35:12,837.10,0.0,12,Valid
2,2026-04-19,13:35:13,824.88,0.0,12,Valid
3,2026-04-19,13:35:14,841.73,0.0,12,Valid
4,2026-04-19,13:35:15,273.36,0.0,12,Valid
5,2026-04-19,13:35:16,832.84,0.0,12,Valid
6,2026-04-19,13:35:17,348.55,0.0,12,Valid
7,2026-04-19,13:35:18,805.25,0.0,12,Valid
8,2026-04-19,13:35:19,805.25,0.0,12,Valid


In [18]:
# @title My Deeper Analysis and Information
# I have:
# -> Obtain precise date and time from $GPRMC system synchronization.
# -> Convert knots to km/h
# -> Determine valid signal reliability
# My observation: Because I use virtual environment to get data so the speed could be unrealistic (But the assignment allows it).